In [ ]:
from dataset import ChessDataset
from model import ChessCNN
from encoder import board_to_tensor
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import math
import time
from tqdm import tqdm

BATCH_SIZE = 250000
TOTAL_SIZE = 12900000

NUM_BATCH = math.ceil(TOTAL_SIZE/BATCH_SIZE)

In [10]:
X_chunks = []
Y_chunks = []

for chunk_id in range(52):
    X_chunks.append(torch.load(fr"D:\Coding\CNN Chess Engine\data\pre-tensors\X_{chunk_id+1}.pt"))
    Y_chunks.append(torch.load(fr"D:\Coding\CNN Chess Engine\data\pre-tensors\Y_{chunk_id+1}.pt"))

X = torch.cat(X_chunks,dim=0)
Y = torch.cat(Y_chunks,dim=0)


In [11]:
device = torch.device("cuda")

model = ChessCNN().to(device)


In [12]:
dataset = TensorDataset(X, Y)

train_size = int(0.98*len(dataset))
val_size = len(dataset) - train_size

train_dataset , val_dataset = random_split(dataset,[train_size,val_size])

In [13]:
def train_one_epoch(model,loader,optimizer,criterion,device):
    
    model.train()

    running_loss = 0.0
    
    for boards,targets in tqdm(loader, desc="Training"):
        boards = boards.float().to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        predictions = model(boards)

        loss = criterion(predictions,targets.unsqueeze(1))
        

        loss.backward()

        optimizer.step()
        
        running_loss+=loss.item()

    avg_loss = running_loss/len(loader)
    
    return avg_loss

def evaluate(model,loader,criterion,device):
    model.eval()

    running_loss =0

    with torch.no_grad():

        for boards,targets in loader:
            boards = boards.float().to(device)
            targets = targets.to(device)

            predictions = model(boards)

            loss = criterion(predictions,targets.unsqueeze(1))

            running_loss +=loss.item()
    
    avg_loss = running_loss/len(loader)

    return avg_loss

In [19]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=0)
test_loader = DataLoader(val_dataset,batch_size=64,shuffle=True,num_workers=0)


In [15]:

optimizer = optim.Adam(model.parameters(),lr=0.001)

criterion = nn.HuberLoss(delta=0.1)

best_val_loss = float("inf")

checkpoint  = torch.load(r"d:\Coding\CNN Chess Engine\data\chess_models\best_model.pth")

best_val_loss = checkpoint["val_loss"]


In [20]:
start_time = time.time()


num_epochs = 100
best_epoch =-1

model.load_state_dict(checkpoint["model_state_dict"])

for epoch in range(num_epochs):
  print(f"Epoch {epoch+1} / {num_epochs}")

  train_loss = train_one_epoch(model,train_loader,optimizer,criterion,device)

  val_loss = evaluate(model,test_loader,criterion,device)

  torch.save(model.state_dict(),fr"d:\Coding\CNN Chess Engine\data\chess_models\1mil\epoch_{epoch+1}.pth")

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    best_epoch = epoch+1
    torch.save({
      "epoch":epoch+1,
      "val_loss":val_loss,
      "train_loss":train_loss,
      "model_state_dict":model.state_dict()},
      r"d:\Coding\CNN Chess Engine\data\chess_models\1mil\best_model.pth")
    print(f"Saved Best Model: {epoch+1}/{num_epochs}")



  print(f"Epoch: {epoch+1}/{num_epochs}")
  print(f"Train Loss: {train_loss:.4f}")
  print(f"Validation Loss: {val_loss:.4f}")

total_time = time.time() - start_time
print(f"\nTotal Training Time: {total_time/60:.2f} minutes")
print(f"Average Epoch Time: {total_time/num_epochs:.2f} seconds")


Epoch 1 / 100


Training: 100%|██████████| 198420/198420 [26:45<00:00, 123.59it/s]


Epoch: 1/100
Train Loss: 0.0114
Validation Loss: 0.0106
Epoch 2 / 100


Training: 100%|██████████| 198420/198420 [30:50<00:00, 107.24it/s]


Epoch: 2/100
Train Loss: 0.0113
Validation Loss: 0.0103
Epoch 3 / 100


...

Saved Best Model: 5/100
Epoch: 5/100
Train Loss: 0.0110
Validation Loss: 0.0099
Epoch 6 / 100


Training: 100%|██████████| 198420/198420 [32:21<00:00, 102.18it/s]


Epoch: 6/100
Train Loss: 0.0109
Validation Loss: 0.0101
Epoch 7 / 100


Training:  59%|█████▉    | 117712/198420 [20:09<13:49, 97.33it/s] 


KeyboardInterrupt: 